In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

In [ ]:
df = session.table("SNOWFLAKE_LEARNING_DB.PUBLIC.NYC_HOURLY_BOROUGH_FINAL").to_pandas()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_ts = df.copy()

df_ts = df.copy()

# clean string columns if they contain extra quotes
string_cols = ["HOUR_TS", "BOROUGH", "SNOW_TEXT", "RAIN_TEXT"]
for col in string_cols:
    if col in df_ts.columns:
        df_ts[col] = df_ts[col].astype(str).str.replace('"', '', regex=False).str.strip()

# convert timestamp
df_ts["HOUR_TS"] = pd.to_datetime(df_ts["HOUR_TS"], errors="coerce")

# drop invalid timestamps
df_ts = df_ts.dropna(subset=["HOUR_TS"])

# sort by time
df_ts = df_ts.sort_values("HOUR_TS")

# optional: clean borough labels
df_ts["BOROUGH"] = df_ts["BOROUGH"].str.upper().str.strip()

print(df_ts[["HOUR_TS", "BOROUGH", "TRIPS"]].head())
print(df_ts.shape)

In [ ]:
numeric_cols = [
    "TRIPS",
    "AVG_FARE_AMOUNT",
    "AVG_TIP_AMOUNT",
    "AVG_TOTAL_AMOUNT",
    "AVG_TRIP_DISTANCE",
    "AVG_TRIP_DURATION_MIN",
    "TEMP_AVG",
    "DEWPOINT_AVG",
    "HUMIDITY_AVG",
    "WIND_AVG",
    "WIND_GUST_AVG",
    "PRESSURE_AVG",
    "PRECIP_TOTAL",
    "SNOW_TOTAL",
    "SNOW_CODE",
    "SNOW",
    "RAIN",
    "RAIN_CODE",
    "EXTREME_WEATHER",
    "COMPLAINTS_311",
    "COMPLAINTS_NOISE",
    "COMPLAINTS_SANITATION",
    "COMPLAINTS_TRANSPORTATION",
    "YEAR",
    "MONTH",
    "DAY",
    "HOUR"
]

for col in numeric_cols:
    if col in df_ts.columns:
        df_ts[col] = pd.to_numeric(df_ts[col], errors="coerce")

In [ ]:
df_ts = df_ts.set_index("HOUR_TS")
df_ts = df_ts.sort_index()

print(df_ts.index.min(), df_ts.index.max())

In [ ]:
df_manhattan = df_ts[df_ts["BOROUGH"] == "MANHATTAN"].copy()
print(df_manhattan.shape)
df_manhattan.head()

In [ ]:
df_manhattan.columns

In [ ]:
df_manhattan["datetime"] = pd.to_datetime(
    df_manhattan[["YEAR", "MONTH", "DAY", "HOUR"]]
)

# sort by time
df_manhattan = df_manhattan.sort_values("datetime").reset_index(drop=True)

# check
display(df_manhattan[["YEAR", "MONTH", "DAY", "HOUR", "datetime", "TRIPS"]].head())
print(df_manhattan["datetime"].min(), "to", df_manhattan["datetime"].max())

In [ ]:
df_manhattan = df_manhattan.set_index("datetime")
df_manhattan.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,5))

plt.plot(df_manhattan.index, df_manhattan["TRIPS"])

plt.title("Manhattan Taxi Trip Demand Over Time")
plt.xlabel("Date")
plt.ylabel("Trips")

plt.show()

In [ ]:
manhattan_daily = df_manhattan.resample("D").agg({
    "TRIPS":"sum"
})

manhattan_daily.head()

In [ ]:
plt.figure(figsize=(14,5))

plt.plot(manhattan_daily.index, manhattan_daily["TRIPS"])

plt.title("Daily Manhattan Taxi Demand")
plt.xlabel("Date")
plt.ylabel("Trips")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# decomposition
decomp = seasonal_decompose(
    manhattan_daily["TRIPS"],
    model="additive",
    period=7
)

trend = decomp.trend
seasonal = decomp.seasonal
resid = decomp.resid
observed = decomp.observed


# nicer plotting
fig, axes = plt.subplots(4, 1, figsize=(14,10), sharex=True)

axes[0].plot(observed, linewidth=1.2)
axes[0].set_title("Observed Trips")
axes[0].grid(alpha=0.3)

axes[1].plot(trend, linewidth=1.2, color="darkorange")
axes[1].set_title("Trend")
axes[1].grid(alpha=0.3)

axes[2].plot(seasonal, linewidth=1.2, color="green")
axes[2].set_title("Weekly Seasonality")
axes[2].grid(alpha=0.3)

axes[3].plot(resid, linewidth=1.2, color="red")
axes[3].set_title("Residual")
axes[3].grid(alpha=0.3)

plt.suptitle("Seasonal Decomposition of Manhattan Taxi Demand", fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

## stationary
result = adfuller(manhattan_daily["TRIPS"])

print("ADF Statistic:", result[0])
print("p-value:", result[1])
print("Lags Used:", result[2])
print("Observations:", result[3])

for key, value in result[4].items():
    print(f"Critical Value ({key}): {value}")

In [ ]:
daily_features = df_manhattan.resample("D").agg({
    "TRIPS": "sum",

    "TEMP_AVG": "mean",
    "DEWPOINT_AVG": "mean",
    "HUMIDITY_AVG": "mean",
    "WIND_AVG": "mean",
    "PRESSURE_AVG": "mean",
    "PRECIP_TOTAL": "sum",
    "EXTREME_WEATHER": "max",

    "COMPLAINTS_NOISE": "sum",
    "COMPLAINTS_SANITATION": "sum",
    "COMPLAINTS_TRANSPORTATION": "sum"
})

daily_features = daily_features.dropna()

daily_features.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

sns.heatmap(
    daily_features.corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Correlation: Trips vs Weather vs Complaints")

plt.show()

In [ ]:
import statsmodels.api as sm

X = daily_features[[
    "TEMP_AVG",
    "WIND_AVG",
    "PRECIP_TOTAL",
    "EXTREME_WEATHER",
    "COMPLAINTS_NOISE",
    "COMPLAINTS_SANITATION",
    "COMPLAINTS_TRANSPORTATION"
]]

y = daily_features["TRIPS"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
y = daily_features["TRIPS"]

exog = daily_features[[
    "TEMP_AVG",
    "PRECIP_TOTAL",
    "EXTREME_WEATHER",
    "COMPLAINTS_TRANSPORTATION"
]]

In [ ]:
seasonal_period = 7

from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    y,
    exog=exog,
    order=(1,0,1),
    seasonal_order=(1,0,1,7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit()

print(results.summary())

In [ ]:
daily_features["predicted"] = results.predict()

import matplotlib.pyplot as plt

plt.figure(figsize=(14,5))

plt.plot(daily_features.index, y, label="Actual Trips")
plt.plot(daily_features.index, daily_features["predicted"], label="Predicted Trips")

plt.title("Actual vs Predicted Taxi Demand (SARIMAX)")
plt.legend()

plt.show()

In [ ]:
forecast = results.get_forecast(
    steps=14,
    exog=exog.tail(14)
)

forecast_values = forecast.predicted_mean

In [ ]:
train_size = int(len(daily_features) * 0.7)

train = daily_features.iloc[:train_size].copy()
test = daily_features.iloc[train_size:].copy()

y_train = train["TRIPS"]
y_test = test["TRIPS"]

exog_vars = [
    "TEMP_AVG",
    "PRECIP_TOTAL",
    "EXTREME_WEATHER",
    "COMPLAINTS_TRANSPORTATION"
]

X_train = train[exog_vars]
X_test = test[exog_vars]

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    y_train,
    exog=X_train,
    order=(1,0,1),
    seasonal_order=(1,0,1,7)
)

results = model.fit(disp=False)

forecast = results.get_forecast(
    steps=len(X_test),
    exog=X_test
)

pred = forecast.predicted_mean
pred.index = y_test.index

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
mape = np.mean(np.abs((y_test - pred) / y_test)) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)